# Fundamentos — baseline sin RAG (3 modelos, corpus en español)

Sub-proyecto 1 del Trabajo Final. Corre GPT-4o mini (OpenAI), LLaMA 3.1 8B y Mistral (ambos vía Ollama local) sobre la muestra estratificada traducida al español, sin RAG.

In [1]:
import sys, os
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv()
from src import framework_tf as ftf
from src import corpus

ftf.verificar_ollama()
print('Ollama OK')

Ollama OK


## 1. Dataset y muestra en español

In [2]:
from datasets import load_dataset
import pandas as pd

N_MUESTRA = 20
ds = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train')
df_full = ds.to_pandas()

from openai import OpenAI
cliente_traductor = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

muestra = corpus.cargar_o_generar_corpus_es(
    '../data/corpus_es_muestra.csv', df_full, N_MUESTRA, cliente_traductor,
)
print(f'Muestra: {len(muestra)} filas, {muestra["category"].nunique()} categorías')
muestra[['category', 'instruction_es', 'response_es']].head(3)

/Users/jxifro/Desktop/Trabajo Final/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Muestra: 20 filas, 11 categorías


,category,instruction_es,response_es
0,ACCOUNT,¿Dónde notificar sobre problemas con el registro?,Nos alegra que te hayas puesto en contacto con...
1,CANCEL,Estoy tratando de ver la penalización por sali...,¡Por supuesto! Para ver la penalización por sa...
2,CONTACT,Tengo que verificar en qué horarios puedo acce...,¡Es un honor ayudar! Estoy al tanto de que que...


## 2. Generación sobre los 3 modelos

In [3]:
from tqdm.auto import tqdm
import time

def correr_experimento(modelo_key, df):
    filas = []
    for i, fila in tqdm(df.iterrows(), total=len(df), desc=modelo_key):
        out = ftf.generar(modelo_key, fila['instruction_es'])
        filas.append({
            'modelo': modelo_key,
            'tipo': ftf.MODELOS[modelo_key]['tipo'],
            'category': fila['category'],
            'consulta': fila['instruction_es'],
            'referencia': fila['response_es'],
            **out,
        })
        if ftf.MODELOS[modelo_key]['proveedor'] == 'openai':
            time.sleep(0.5)
    return pd.DataFrame(filas)

resultados = pd.concat(
    [correr_experimento(m, muestra) for m in ftf.MODELOS],
    ignore_index=True,
)
n_err = resultados['error'].notna().sum()
print(f'Completadas: {len(resultados) - n_err}/{len(resultados)}')
if n_err:
    print(resultados[resultados['error'].notna()][['modelo', 'error']])

gpt-4o-mini:   0%|          | 0/20 [00:00<?, ?it/s]

gpt-4o-mini:   5%|▌         | 1/20 [00:02<00:43,  2.31s/it]

gpt-4o-mini:  10%|█         | 2/20 [00:03<00:33,  1.86s/it]

gpt-4o-mini:  15%|█▌        | 3/20 [00:05<00:31,  1.86s/it]

gpt-4o-mini:  20%|██        | 4/20 [00:06<00:25,  1.57s/it]

...

mistral:  90%|█████████ | 18/20 [02:38<00:15,  7.78s/it]

mistral:  95%|█████████▌| 19/20 [02:52<00:09,  9.45s/it]

mistral: 100%|██████████| 20/20 [03:04<00:00, 10.42s/it]

mistral: 100%|██████████| 20/20 [03:04<00:00,  9.25s/it]

Completadas: 60/60


## 3. Métricas de calidad (BERTScore en español + ROUGE-L)

In [4]:
from bert_score import score as bertscore
from rouge_score import rouge_scorer

ok = resultados[resultados['error'].isna()].copy()

P, R, F1 = bertscore(cands=ok['respuesta'].tolist(), refs=ok['referencia'].tolist(),
                     lang='es', verbose=False)
ok['bertscore_f1'] = F1.numpy().round(4)

rs = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
ok['rouge_l'] = [round(rs.score(ref, cand)['rougeL'].fmeasure, 4)
                 for ref, cand in zip(ok['referencia'], ok['respuesta'])]

ok.groupby('modelo')[['bertscore_f1', 'rouge_l', 'latencia_s']].mean().round(4)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11018.26it/s]


[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,bertscore_f1,rouge_l,latencia_s
modelo,,,
gpt-4o-mini,0.7120,0.2301,1.3016
llama-3.1-8b,0.7154,0.2381,6.3767
mistral,0.7103,0.2355,9.2456


## 4. Costo dual e índice compuesto

In [5]:
import numpy as np

costos = ok.apply(lambda f: pd.Series(ftf.costo_consulta(f)), axis=1)
ok = pd.concat([ok, costos], axis=1)
ok['costo_oficial_usd'] = ok.apply(ftf.costo_oficial, axis=1)

def normalizar(s, invertir=False):
    rango = s.max() - s.min()
    if rango == 0:
        return pd.Series(0.5, index=s.index)
    n = (s - s.min()) / rango
    return 1 - n if invertir else n

ok['n_calidad'] = normalizar(ok['bertscore_f1'])
ok['n_costo'] = normalizar(ok['costo_oficial_usd'], invertir=True)
ok['n_latencia'] = normalizar(ok['latencia_s'], invertir=True)
ok['indice'] = ((ok['n_calidad'] + ok['n_costo'] + ok['n_latencia']) / 3).round(4)

resumen = ok.groupby('modelo').agg(
    n=('category', 'count'),
    bertscore_f1=('bertscore_f1', 'mean'),
    rouge_l=('rouge_l', 'mean'),
    latencia_s=('latencia_s', 'mean'),
    costo_electricidad_usd=('costo_electricidad_usd', 'mean'),
    costo_instancia_usd=('costo_instancia_usd', 'mean'),
    costo_oficial_usd=('costo_oficial_usd', 'mean'),
    indice=('indice', 'mean'),
).round(6)
resumen

,n,bertscore_f1,rouge_l,latencia_s,costo_electricidad_usd,costo_instancia_usd,costo_oficial_usd,indice
modelo,,,,,,,,
gpt-4o-mini,20,0.712005,0.230080,1.30155,NaN,NaN,0.000071,0.797245
llama-3.1-8b,20,0.715360,0.238120,6.37675,0.000005,0.001328,0.001328,0.562035
mistral,20,0.710280,0.235515,9.24555,0.000008,0.001926,0.001926,0.424655


## 5. Guardado

In [6]:
from datetime import datetime
sello = datetime.now().strftime('%Y%m%d_%H%M')
archivo = f'../resultados_fundamentos_{sello}.csv'
ok.to_csv(archivo, index=False)
print(f'Guardado: {archivo} ({len(ok)} filas)')

Guardado: ../resultados_fundamentos_20260913_1422.csv (60 filas)
